In [1]:
import pyspark

In [ ]:
from google.colab import files

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder\
    .appName("Uber_analysis")\
        .getOrCreate()

In [6]:
df =spark.read.csv('dataset.csv',header =True,inferSchema=True)

In [7]:
from pyspark.sql.functions import *
from pyspark.sql import *


In [8]:
df_updated = df.withColumn('row_id',monotonically_increasing_id())


In [9]:
#Create window specification
windowSpec = Window.orderBy("row_id").rowsBetween(Window.unboundedPreceding, 0)

# Forward fill the Date column
df_updated = df_updated.withColumn(
    "Date",
    last(col("Date"), ignorenulls=True).over(windowSpec)
)


df_updated = df_updated.drop('row_id')

In [10]:
import pyspark.sql.functions as F

What's the average number of unique drivers per day?

In [11]:
df_updated.show()

+---------+------------+---------+-------+----------------+---------+--------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|
+---------+------------+---------+-------+----------------+---------+--------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|
|10-Sep-12|           8|        6|      0|               2|        2|            14|
|10-Sep-12|           9|        8|      3|               0|        0|            14|
|10-Sep-12|          10|        9|      2|               0|        1|            14|
|10-Sep-12|          11|       11|      1|               4|        4|            11|
|10-Sep-12|          12|       12|      0|               2|        2|            11|
|10-Sep-12|          13|        9|      1|               0|        0|             9|
|10-Sep-12|          14|       12|      1|               0|        0|             9|
|10-Sep-12|          15|       11|      2|               1|      

In [12]:
df_num_unique = df_updated.groupBy('Date').agg(F.round(F.avg('Unique Drivers'),2).alias('Average Drivers'))

In [13]:

df_num_unique.show()

+---------+---------------+
|     Date|Average Drivers|
+---------+---------------+
|10-Sep-12|           8.12|
|11-Sep-12|            5.5|
|12-Sep-12|           8.38|
|13-Sep-12|           6.63|
|14-Sep-12|           8.79|
|15-Sep-12|           8.04|
|16-Sep-12|           6.25|
|17-Sep-12|           6.83|
|18-Sep-12|           5.58|
|19-Sep-12|           7.75|
|20-Sep-12|           7.46|
|21-Sep-12|          12.42|
|22-Sep-12|          12.17|
|23-Sep-12|            8.5|
|24-Sep-12|           1.71|
+---------+---------------+



Which date had the most unique drivers available?

In [14]:
df_most_uniq = df_updated.groupBy('Date').agg((F.sum('Unique Drivers')).alias('Max Drivers'))

In [15]:
df_most_uniq.show()

+---------+-----------+
|     Date|Max Drivers|
+---------+-----------+
|10-Sep-12|        138|
|11-Sep-12|        132|
|12-Sep-12|        201|
|13-Sep-12|        159|
|14-Sep-12|        211|
|15-Sep-12|        193|
|16-Sep-12|        150|
|17-Sep-12|        164|
|18-Sep-12|        134|
|19-Sep-12|        186|
|20-Sep-12|        179|
|21-Sep-12|        298|
|22-Sep-12|        292|
|23-Sep-12|        204|
|24-Sep-12|         12|
+---------+-----------+



In [16]:
df_most_uniq.orderBy(df_most_uniq['Max Drivers'].desc()).limit(1).show()

+---------+-----------+
|     Date|Max Drivers|
+---------+-----------+
|21-Sep-12|        298|
+---------+-----------+




Calculate the ratio of completed trips to requests (completion rate)


In [17]:
df_ratio = df_updated.withColumn('ratio',F.when(col('Requests ')==0 ,0).otherwise(col('Completed Trips ')/col('Requests ')*100))

In [18]:
df_ratio.show()

+---------+------------+---------+-------+----------------+---------+--------------+-----------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|            ratio|
+---------+------------+---------+-------+----------------+---------+--------------+-----------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|            100.0|
|10-Sep-12|           8|        6|      0|               2|        2|            14|            100.0|
|10-Sep-12|           9|        8|      3|               0|        0|            14|              0.0|
|10-Sep-12|          10|        9|      2|               0|        1|            14|              0.0|
|10-Sep-12|          11|       11|      1|               4|        4|            11|            100.0|
|10-Sep-12|          12|       12|      0|               2|        2|            11|            100.0|
|10-Sep-12|          13|        9|      1|               0|        0|    

In [19]:
df_ratio=df_ratio.groupBy('Date').agg(F.round(F.avg('ratio'),2).alias('Completion Rate'))


In [20]:
df_ratio.show()

+---------+---------------+
|     Date|Completion Rate|
+---------+---------------+
|10-Sep-12|          55.39|
|11-Sep-12|          46.93|
|12-Sep-12|          64.55|
|13-Sep-12|          54.25|
|14-Sep-12|          68.59|
|15-Sep-12|          64.65|
|16-Sep-12|          64.19|
|17-Sep-12|          54.61|
|18-Sep-12|          46.59|
|19-Sep-12|          57.01|
|20-Sep-12|          57.07|
|21-Sep-12|          58.27|
|22-Sep-12|          68.63|
|23-Sep-12|          66.16|
|24-Sep-12|          35.71|
+---------+---------------+



In [21]:
df_ratio.agg(F.avg('Completion Rate').alias('Ratio')).select('Ratio').first()['Ratio']

57.50666666666667

Find hours with the highest "Zeroes" (supply shortage)'

In [22]:
df_most_zeros = df_updated.groupby('Time (Local)').agg(F.sum('Zeroes ').alias('Most zero'))

In [23]:
df_most_zeros.show()

+------------+---------+
|Time (Local)|Most zero|
+------------+---------+
|          12|       41|
|          22|       81|
|           1|       43|
|          13|       67|
|          16|       61|
|           6|       32|
|           3|       30|
|          20|       61|
|           5|       31|
|          19|       96|
|          15|       70|
|           9|       39|
|          17|       74|
|           4|       21|
|           8|       30|
|          23|      193|
|           7|       30|
|          10|       46|
|          21|       68|
|          11|       47|
+------------+---------+
only showing top 20 rows


In [24]:
df_most_zeros.orderBy('Time (Local)',ascending = False).limit(1).show()

+------------+---------+
|Time (Local)|Most zero|
+------------+---------+
|          23|      193|
+------------+---------+



Efficiency

identify peak demand hours (highest Requests)

In [25]:
df_updated.show()

+---------+------------+---------+-------+----------------+---------+--------------+
|     Date|Time (Local)|Eyeballs |Zeroes |Completed Trips |Requests |Unique Drivers|
+---------+------------+---------+-------+----------------+---------+--------------+
|10-Sep-12|           7|        5|      0|               2|        2|             9|
|10-Sep-12|           8|        6|      0|               2|        2|            14|
|10-Sep-12|           9|        8|      3|               0|        0|            14|
|10-Sep-12|          10|        9|      2|               0|        1|            14|
|10-Sep-12|          11|       11|      1|               4|        4|            11|
|10-Sep-12|          12|       12|      0|               2|        2|            11|
|10-Sep-12|          13|        9|      1|               0|        0|             9|
|10-Sep-12|          14|       12|      1|               0|        0|             9|
|10-Sep-12|          15|       11|      2|               1|      

In [26]:
df_peak_demand = df_updated.groupBy('Time (Local)').agg(F.sum('Requests ').alias('Peak Demand'))

In [27]:
df_peak_demand.show()

+------------+-----------+
|Time (Local)|Peak Demand|
+------------+-----------+
|          12|         53|
|          22|        174|
|           1|         96|
|          13|         55|
|          16|         82|
|           6|         28|
|           3|         35|
|          20|        107|
|           5|         14|
|          19|        156|
|          15|         71|
|           9|         26|
|          17|         98|
|           4|          9|
|           8|         29|
|          23|        184|
|           7|         22|
|          10|         28|
|          21|        112|
|          11|         47|
+------------+-----------+
only showing top 20 rows


In [28]:
df_peak_demand.orderBy(df_peak_demand['Peak Demand'].desc()).limit(1).show()

+------------+-----------+
|Time (Local)|Peak Demand|
+------------+-----------+
|          23|        184|
+------------+-----------+



Find the correlation between Eyeballs and Completed Trips

In [31]:

df_corr = df_updated.agg(F.corr('Eyeballs ', 'Completed Trips ').alias("correlation"))


correlation = df_corr.first()["correlation"]
print(f"Correlation between Eyeballs and Completed Trips: {correlation:.2f}")

Correlation between Eyeballs and Completed Trips: 0.88


In [ ]:
# Your code is correct! Just extract the value
df_corr = df_updated.agg(F.corr('Eyeballs ', 'Completed Trips ').alias("correlation"))

# Extract the value
correlation = df_corr.first()["correlation"]
print(f"Correlation between Eyeballs and Completed Trips: {correlation:.2f}")

Calculate average trips per driver per day



In [41]:
df_avg_trip_per_day = df_updated.groupBy('Date').agg((F.sum('Completed Trips ')/F.avg('Unique Drivers')).alias('Average Trips per Driver'))

In [42]:
df_avg_trip_per_day.show()

+---------+------------------------+
|     Date|Average Trips per Driver|
+---------+------------------------+
|10-Sep-12|      3.2028985507246377|
|11-Sep-12|      7.2727272727272725|
|12-Sep-12|      10.865671641791044|
|13-Sep-12|      6.7924528301886795|
|14-Sep-12|       12.28436018957346|
|15-Sep-12|       24.74611398963731|
|16-Sep-12|                   14.88|
|17-Sep-12|       8.341463414634147|
|18-Sep-12|       7.522388059701493|
|19-Sep-12|       5.290322580645161|
|20-Sep-12|       9.385474860335195|
|21-Sep-12|      15.302013422818792|
|22-Sep-12|      20.383561643835616|
|23-Sep-12|      13.058823529411764|
|24-Sep-12|      2.3333333333333335|
+---------+------------------------+



In [45]:
final_Avg  = df_avg_trip_per_day.agg(F.avg('Average Trips per Driver').alias('Average Trips')).first()['Average Trips']

In [47]:

print(f"Average trips per driver per day: {final_Avg:.2f}")

Average trips per driver per day: 10.78
